[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/pytorch/lab-p5-boundary-trace.ipynb)

# LAB·P5 · The boundary trace

**Hardware:** Colab TPU. Runtime → Change runtime type → TPU before anything else. Everything here also runs on a linux box with the CPU plugin, where `PJRT_DEVICE=CPU` is the only change; the counters are identical and only the device string moves.

**Version:** every source claim below was read against `pytorch/xla` at commit `41398bf`, which `setup.py:164` stamps as version `2.9.0`. The install cell pins that pair. Counter names are not API, so if you run a different torch_xla, expect a name or two to have drifted, and trust your own report over this text.

Nothing in this notebook has been run. It was authored on a machine with no torch_xla wheel, so every output cell is empty and every place a number belongs carries the marker `captured on Colab: pending` next to a blank. The first real numbers this lab holds will be yours.

## What you are tracing

A tensor on an `xla` device does not compute when you write the op. It records. Somewhere later a graph gets cut, lowered, compiled, and run, and everything between your Python and the accelerator passes through one C++ class: `ComputationClient`, pure virtual, declared at `torch_xla/csrc/runtime/computation_client.h:55`. One implementation, `PjRtComputationClient`, is the only code in torch_xla that ever calls PJRT. Chapter 10 argues that shape in prose. This lab checks it.

You will run the same training step three ways, then read what each one did to the boundary:

1. **Lazy**, the default. Ops accumulate, `torch_xla.sync()` cuts the graph, one compile and one execute follow.
2. **Eager**, `torch_xla.experimental.eager_mode(True)`. The same tracing machinery with the barrier moved to after every op, so the counters go up per op rather than per step.
3. **Dynamo**, `torch.compile(backend="openxla")`. Dynamo captures FX graphs ahead of time and the bridge dispatches by a precomputed hash.

Three instruments record what happened. The metrics report counts every seam call by name. The IR and HLO dumps write down what crossed. And `TF_CPP_VMODULE` turns on the log lines that `PjRtComputationClient` prints from inside the calls themselves.

## The seam table

Read this before you run anything. Left column is a name that will appear in your metrics report. The rest is where that name is incremented and what it means happened underneath. Line numbers are `torch_xla/csrc/runtime/pjrt_computation_client.cpp` at `41398bf` unless another file is named.

| report name | ComputationClient method | bottoms out in | cited |
|---|---|---|---|
| `CompileTime` | `Compile` | `client_->CompileAndLoad(computation, options)` | `:546`, `:659` |
| `EagerOpCompileTime` | `Compile`, with `eager_mode` set on the instance | the same `CompileAndLoad` | `:546-549` |
| `CreateCompileHandles` | `Compile` | one per program that came back loaded | `:683` |
| `StableHloCompile` | `Compile` under `XLA_STABLEHLO_COMPILE` | `CompileAndLoad` over an `mlir::ModuleOp` instead of an HLO proto | `:641`, `:655` |
| `ExecuteTime` | `ExecuteComputation` | `executable->ExecuteSharded(buffers, device, opts, &future)` | `:745`, `:786` |
| `EagerOpExecuteTime` | `ExecuteComputation`, eager | the same `ExecuteSharded` | `:745-749` |
| `ExecuteReplicatedTime` | `ExecuteReplicated` | `executable->Execute(handles, opts, &futures)` | `:821`, `:889` |
| `TransferToDeviceTime`, `OutboundData` | `TransferToDevice` | `client_->BufferFromHostBuffer(...)` | `:267`, `:279`, `:294` |
| `TransferFromDeviceTime`, `InboundData` | `TransferFromDevice` | `buffer->ToLiteral(&literal)` then `xla::JoinFutures(...).Await()` | `:516`, `:534-539` |
| `CreateDataHandles` | anything that hands back a `PjRtData` | no PJRT call of its own; the handles are counted as they are built | `:295`, `:805` |

Four more names show up in the same report and never touch the seam. They are frontend bookkeeping, and telling them apart from the rows above is most of the skill this lab is after.

| report name | what raises it | cited |
|---|---|---|
| `MarkStep` | `XLAGraphExecutor::MarkStep`, once per step barrier | `xla_graph_executor.cpp:447` |
| `CachedCompile` / `UncachedCompile` | the computation-cache lookup, hit and miss | `xla_graph_executor.cpp:1240`, `:1236` |
| `DynamoExtractCompiledGraph` | the dynamo bridge, once per FX graph it turns into a program | `_dynamo/dynamo_bridge.py:671` |
| `DynamoSyncInputExecuteTime` | the blocking sync the bridge does when an input still carries pending IR | `_dynamo/dynamo_bridge.py:523` |

The split is not a convention someone chose to be tidy. `_xla_metrics_report` stitches two independent reports together, one from `torch::lazy` and one from `client->GetMetrics()`, with a comment saying the two sets should not overlap because `ComputationClient` cannot depend on PyTorch (`init_python_bindings.cpp:2464-2480`). The report you are about to print has the seam running down the middle of it.

**your prediction:** one lazy training step, one `torch_xla.sync()`, one `.item()` at the end. Which rows in the first table go up, and by how much? Write the numbers down before you run anything.

## Install

torch and torch_xla ship as a matched pair and the versions must agree. This lab was written against `2.9.0`. If that pair has aged out, check github.com/pytorch/xla for the current one and change both numbers together.

In [ ]:
# Colab TPU runtime only
!pip install -q torch==2.9.0 'torch_xla[tpu]==2.9.0'

In [ ]:
# Colab TPU runtime only · if the import fails, Runtime -> Restart session, then rerun
# this cell alone. The install above does not need to run twice.
#
# One rule holds for the whole lab: a TPU chip admits one process. Every trace
# below runs as its own subprocess, so this kernel must not touch the device
# until the coda at the end. Import, print versions, query nothing.
import os
from importlib.metadata import version

import torch
import torch_xla
import torch_xla.runtime as xr
import torch_xla.debug.metrics as met

print("torch", torch.__version__, "| torch_xla", version("torch_xla"))

## The same step, in a script

The next two traces need a clean process each. Eager mode is a process-wide bool on the graph executor, the dump flags are read when the extension initializes, and dynamo's caches survive nothing you can clear from a notebook cell. Comparing three modes inside one kernel would compare three different kernel histories.

There is a harder constraint underneath those: the chip itself. A TPU admits one process, so the moment this kernel touches the device, every subprocess after it dies with "The TPU is already in use". The kernel therefore stays off the chip for the whole lab, and every trace, the first one included, runs as its own process.

So the step moves into a file. One script, one mode argument, the same model and the same shapes every time, and a compact row of seam counters printed at the end so the three runs can be lined up side by side.

In [ ]:
# Colab TPU runtime only
script = r'''
import json
import sys

import torch
import torch.nn as nn
import torch_xla
import torch_xla.debug.metrics as met
from torch_xla.experimental import eager_mode

# Left column of the seam table: every one of these is raised inside
# PjRtComputationClient, on the far side of the boundary.
SEAM = [
    "CompileTime", "EagerOpCompileTime",
    "ExecuteTime", "EagerOpExecuteTime", "ExecuteReplicatedTime",
    "TransferToDeviceTime", "TransferFromDeviceTime",
    "InboundData", "OutboundData",
]
# Frontend bookkeeping: raised before the seam, by torch_xla itself.
FRONTEND = [
    "MarkStep", "CachedCompile", "UncachedCompile",
    "CreateDataHandles", "CreateCompileHandles", "StableHloCompile",
    "DynamoExtractCompiledGraph", "DynamoSyncInputExecuteTime",
]


def seam_row(mode):
    # One flat dict: how many times each named thing happened.
    row = {"mode": mode}
    for name in SEAM:
        data = met.metric_data(name)          # (TOTAL_SAMPLES, ACCUMULATOR, SAMPLES)
        row[name] = None if data is None else data[0]
    for name in FRONTEND:
        row[name] = met.counter_value(name)   # None when the counter never fired
    return row


def train_step(model, opt, x, y):
    loss = nn.functional.mse_loss(model(x), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    return loss


mode = sys.argv[1]
steps = int(sys.argv[2]) if len(sys.argv) > 2 else 1

if mode == "eager":
    eager_mode(True)   # before anything is built, per the eager docs

dev = torch_xla.device()
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 128)).to(dev)
opt = torch.optim.SGD(model.parameters(), lr=1e-2)
x = torch.randn(32, 128).to(dev)
y = torch.randn(32, 128).to(dev)

step = torch.compile(train_step, backend="openxla") if mode == "dynamo" else train_step

torch_xla.sync()
met.clear_all()

for _ in range(steps):
    loss = step(model, opt, x, y)
    if mode == "lazy":
        torch_xla.sync()   # eager ran every op already; the dynamo bridge syncs itself
final = loss.item()

print(met.metrics_report())
print("SEAM_ROW " + json.dumps(seam_row(mode)))
'''

with open("boundary_step.py", "w") as f:
    f.write(script)
print("wrote boundary_step.py")

## Trace one: lazy

This trace runs the script you will write below in its own process (write the script first; the cells are ordered that way). The step is deliberately small: one MLP, one MSE loss, one SGD update, fixed shapes. Small enough that nothing in the report is noise, big enough that the compiler has real work.

Two lines carry the whole method. The `torch_xla.sync()` before `met.clear_all()` flushes everything that building the model and moving the tensors queued up, so the counters that follow belong to the step and to nothing else. The `.item()` at the end is the read that forces the result home; without it the step would still be in flight when the report printed.

In [ ]:
# Colab TPU runtime only · trace one runs in its own process, like every trace here
import subprocess
import sys

r = subprocess.run(
    [sys.executable, "boundary_step.py", "lazy", "1"],
    capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("trace one failed; see stderr above")

## Read your report against the table

The report prints metrics as `TotalSamples` plus an accumulator plus a percentile block, and counters as a single value. For the seam question, `TotalSamples` is the number that matters: it is how many times that call was made, and the timings are a separate question you can come back to.

Work down the report and answer four things in writing.

- How many times did `CompileTime` fire, and does `CachedCompile` plus `UncachedCompile` account for it? A cache hit skips the seam entirely; only the miss reaches `Compile`.
- `ExecuteTime` or `ExecuteReplicatedTime`? Non-SPMD single-device work takes `ExecuteComputation` and lands on `ExecuteSharded`; SPMD takes the replicated path. Your report says which world you are in without you having to guess.
- Which side of the report holds `TransferFromDeviceTime`, and what forced it? Nothing in the step asked for a value. The `.item()` did.
- `OutboundData` and `InboundData` are byte counts. Compare outbound against the parameters and inputs you moved with `.to(dev)`, and inbound against one scalar loss.

**captured on Colab: pending**

```
CompileTime TotalSamples:
UncachedCompile:
CachedCompile:
ExecuteTime TotalSamples:
TransferToDeviceTime TotalSamples:
TransferFromDeviceTime TotalSamples:
OutboundData (bytes):
InboundData (bytes):
MarkStep:
```

## Traces two and three

**your prediction, eager:** the step has roughly a dozen ops in it. Eager mode compiles and executes each one on its own. Does `CompileTime` go up a dozen times, or does a different name carry it? And what does the second identical step cost, given that single-op programs go into the same computation cache as everything else?

**your prediction, dynamo:** `openxla` is registered in PyTorch as `aot_autograd(fw_compiler=openxla_eval_boxed)`, which means forward and backward arrive at the bridge as separate FX graphs. Run two steps. How many compiles, and how many executes per step after the first?

Two steps each, so the second one shows what steady state costs.

In [ ]:
# Colab TPU runtime only
import json
import subprocess
import sys

rows = {}
for mode in ["lazy", "eager", "dynamo"]:
    print("=" * 72)
    print(mode)
    print("=" * 72)
    r = subprocess.run(
        [sys.executable, "boundary_step.py", mode, "2"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"{mode} trace failed; see stderr above")
    out = r.stdout
    print(out)
    for line in out.splitlines():
        if line.startswith("SEAM_ROW "):
            rows[mode] = json.loads(line[len("SEAM_ROW "):])

In [ ]:
# Colab TPU runtime only · the three-way diff, one row per name
names = [k for k in rows["lazy"] if k != "mode"]
width = max(len(n) for n in names)
print(f"{'':{width}}  {'lazy':>8} {'eager':>8} {'dynamo':>8}")
for name in names:
    cells = "".join(f"{str(rows[m].get(name)):>9}" for m in ["lazy", "eager", "dynamo"])
    print(f"{name:{width}}  {cells}")

## Read the three-way diff

The interesting cells are the empty ones. A name that is `None` in one column and a number in another is a path that mode never took.

- Eager reports under `EagerOpCompileTime` and `EagerOpExecuteTime` rather than `CompileTime` and `ExecuteTime`. Nothing about the compilation changed. `instances.front().eager_mode` travels with the compile instance and `PjRtComputationClient` picks a different metric object for the same `CompileAndLoad` call (`pjrt_computation_client.cpp:546-549`, `:745-749`). Same seam call, different name on the tally sheet.
- Compare eager's compile count against its execute count on step two. Single-op programs go into the same LRU computation cache as step graphs, so a repeated op shape is a cache hit and compiles stop while executes keep going.
- Dynamo's `CompileTime` should be small and flat after the first step, and `DynamoExtractCompiledGraph` tells you how many separate FX graphs the bridge turned into programs. The upstream docs put the count at three per training step against one for lazy, forward and backward and a sync-input graph.
- `MarkStep` counts step barriers, and only the lazy loop calls one on purpose. Whatever eager and dynamo report on that row, they raised themselves. It is the plainest statement of the difference between the three modes that the runtime offers.

**captured on Colab: pending**

```
                        lazy     eager    dynamo
CompileTime
EagerOpCompileTime
ExecuteTime
EagerOpExecuteTime
MarkStep
CachedCompile
UncachedCompile
DynamoExtractCompiledGraph
```

## What crossed: the IR dump and the HLO dump

Counting calls says how often the boundary was crossed. The dumps say what went over it.

Three environment variables, all documented in `docs/source/learn/troubleshoot.md`, and all read at process start, which is the other reason the step lives in a script:

- `XLA_IR_DEBUG=1` captures the Python stack where each IR node was created, and `XLA_HLO_DEBUG=1` propagates that frame into the HLO metadata. Set both, and every instruction in the dump can be traced back to the line of Python that made it.
- `XLA_SAVE_TENSORS_FILE` is the path torch_xla appends the graph to at each execution. One catch the docs do not stress: the runtime appends the device ordinal to whatever path you give it (`GetEnvOrdinalPath`, `sys_util.h`, with a test asserting `/path/to/test/data.42`), so asking for `/tmp/trace.ir` produces `/tmp/trace.ir.0` on a single-device run, one file per device under multiprocessing. The cells below glob for the suffixed name, and `XLA_SAVE_TENSORS_FMT` picks the form: `text` for the lazy IR, `hlo` for the lowered module, `dot` for a graph you can render. This is the frontend's own record, written on the near side of the seam.
- `XLA_FLAGS=--xla_dump_to=/tmp/hlo-dump` is the compiler's record, written on the far side. Same program, one file per compilation, after `CompileAndLoad` handed it to XLA.

The file is appended to rather than replaced, so each run starts by deleting last run's copy.

In [ ]:
# Colab TPU runtime only
import os
import shutil
import subprocess
import sys

import glob

for path in glob.glob("/tmp/trace.ir*") + glob.glob("/tmp/trace.hlo*"):
    os.remove(path)
shutil.rmtree("/tmp/hlo-dump", ignore_errors=True)

runs = {
    "ir": {"XLA_IR_DEBUG": "1", "XLA_HLO_DEBUG": "1",
           "XLA_SAVE_TENSORS_FMT": "text", "XLA_SAVE_TENSORS_FILE": "/tmp/trace.ir"},
    "hlo": {"XLA_IR_DEBUG": "1", "XLA_HLO_DEBUG": "1",
            "XLA_SAVE_TENSORS_FMT": "hlo", "XLA_SAVE_TENSORS_FILE": "/tmp/trace.hlo"},
    "compiler": {"XLA_FLAGS": "--xla_dump_to=/tmp/hlo-dump"},
}
for label, extra in runs.items():
    r = subprocess.run(
        [sys.executable, "boundary_step.py", "lazy", "1"],
        env=dict(os.environ, **extra), capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"{label} run failed; see stderr above")
    print(label, "done")

# the runtime appended the device ordinal, so the files are trace.ir.0 etc.
ir_file = glob.glob("/tmp/trace.ir*")[0]
hlo_file = glob.glob("/tmp/trace.hlo*")[0]
print()
print(ir_file, " ", os.path.getsize(ir_file), "bytes")
print(hlo_file, " ", os.path.getsize(hlo_file), "bytes")
print("/tmp/hlo-dump  ", len(os.listdir("/tmp/hlo-dump")), "files")
for name in sorted(os.listdir("/tmp/hlo-dump"))[:8]:
    print("   ", name)

In [ ]:
# Colab TPU runtime only · the head of each record
import glob

for path in sorted(glob.glob("/tmp/trace.ir*")) + sorted(glob.glob("/tmp/trace.hlo*")):
    print("=" * 72)
    print(path)
    print("=" * 72)
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= 45:
                break
            print(line.rstrip())
    print()

## Read the two records

`DebugUtil::SaveTensorsGraphInfo` writes the file and its shape is fixed (`debug_util.cpp:165`). `[ScheduleSyncTensorsGraph]` names the caller that triggered the dump. `TensorsGraphInfo:` lists the Python frames that built the graph. `Root Hashes: (...)` gives one hash per output. Everything from `## BEGIN_GRAPH` to `## END_GRAPH` is the graph, and a `Graph Hash:` line closes it, appended separately by `SaveGraphHash` (`:189`).

Find four things.

- The `Graph Hash:` line. Copy it down. That hash is what the computation cache is keyed on, so it is the value `CachedCompile` and `UncachedCompile` were voting about, and it is the reason an unchanged step compiles once and runs many times.
- The frames under `TensorsGraphInfo:`. They are your own Python, with `torch_xla.sync` at the bottom of the stack. Change the trigger and this block changes with it, which is how you find an unintended cut point.
- The count of entries in `Root Hashes:` against the number of tensors your step actually updated. A step barrier syncs every live tensor on the device, not only the ones you named.
- One line of the text IR. The form is `%n = <shape> <op>(%operands), tag=value, ROOT=k` (`ir_dump_util.cpp:235-253`). Read the post order and you are reading the graph in the order the lowering will walk it.

Now the `hlo` version of the same graph. This is what `Compile` was handed. With `XLA_HLO_DEBUG=1` set, each instruction carries `metadata` with an `op_type`, an `op_name`, and the file and line of the Python that produced it (`lowering_context.cpp:66-106`). Pick one instruction in the middle of the module and walk it back to a line in your step. That walk is the frontend's whole job made visible.

Last, the compiler's dump directory, which is a different record entirely. `before_optimizations` is what `CompileAndLoad` received; the optimized module beside it is what runs. LAB·X1 reads that pipeline pass by pass. Here it is enough that both records exist because one `Compile` call crossed the seam.

**captured on Colab: pending**

```
graph hash:
root hash count:
IR node count:
one hlo instruction and the python line its metadata names:
files in /tmp/hlo-dump:
```

## The seam, printing from inside itself

Counters and dumps are both inference: you read a tally or a file and conclude a call was made. `PjRtComputationClient` will tell you directly. The calls carry `TF_VLOG` lines, and `TF_CPP_VMODULE` turns them on per source file.

Two of them are worth the noise, both quoted here from the source so you know what to look for rather than what to hope for:

- `"Executing PjRt computation on " << device` at `pjrt_computation_client.cpp:752`, printed at the top of `ExecuteComputation`, before the device lock and before `ExecuteSharded`.
- `"Returning " << datas.size() << " results"` at `:807`, printed after the results have been wrapped as `PjRtData` and before they are handed back, still unready.

At level 3 you also get `"ExecuteComputation returned_future->OnReady finished"` (`:792`), which is the async completion firing on a different thread than the one that made the call. Watching those three lines interleave is the async execution model showing itself.

**your prediction:** how many `Executing PjRt computation` lines for two lazy steps?

In [ ]:
# Colab TPU runtime only
import os
import subprocess
import sys

env = dict(
    os.environ,
    TF_CPP_MIN_LOG_LEVEL="0",
    TF_CPP_VMODULE="pjrt_computation_client=3",
)
proc = subprocess.run(
    [sys.executable, "boundary_step.py", "lazy", "2"],
    env=env, capture_output=True, text=True,
)
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise RuntimeError("log run failed; see stderr above")

WANTED = ("Executing PjRt computation on", "Returning", "OnReady finished")
seam_lines = [l for l in proc.stderr.splitlines() if any(w in l for w in WANTED)]

print(f"{len(proc.stderr.splitlines())} log lines total, {len(seam_lines)} from the seam calls")
print()
for line in seam_lines:
    print(line)

## Reconcile

You now have the same fact recorded three ways: a counter, a dump, and a log line. They should agree, and where they do not, one of the three is counting something you have not accounted for yet.

- Count the `Executing PjRt computation` lines and compare against `ExecuteTime TotalSamples` from the same two-step run. A mismatch means something executed that you did not attribute to a step.
- Compare `CompileTime TotalSamples` against the number of modules in the compiler dump directory. A compile that reached XLA left a file behind.
- Find where the `OnReady finished` line lands relative to the `Returning N results` line. `ExecuteComputation` returns `DataPtr`s that are not ready yet (`:797-804`); the future fires later. If those two lines are adjacent, the device finished before Python got back, and if they are far apart, you have caught the pipeline actually being asynchronous.

**captured on Colab: pending**

```
Executing PjRt computation lines (2 steps):
ExecuteTime TotalSamples (2 steps):
CompileTime TotalSamples:
modules in /tmp/hlo-dump:
OnReady finished, before or after the next step's Executing line:
```

## Coda: the same step, in this kernel

Everything above kept this kernel off the chip so the subprocesses could have it. Now take it. The cell below runs trace one inline, in this process, so you can watch the counters land from code you just typed rather than from a script. After it runs, this kernel holds the TPU, and no subprocess cell above can run again until Runtime -> Restart session. Run it last, and rerun the report reading against the table one more time on its output.

In [ ]:
# Colab TPU runtime only · run LAST: this claims the chip for the kernel
import torch.nn as nn

dev = torch_xla.device()
torch.manual_seed(0)

model = nn.Sequential(nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 128)).to(dev)
opt = torch.optim.SGD(model.parameters(), lr=1e-2)
x = torch.randn(32, 128).to(dev)
y = torch.randn(32, 128).to(dev)

torch_xla.sync()      # flush construction, so the counters below are the step's alone
met.clear_all()

loss = nn.functional.mse_loss(model(x), y)
opt.zero_grad()
loss.backward()
opt.step()
torch_xla.sync()      # the cut point: lower, compile, execute
final = loss.item()   # the read that forces the transfer home

print(f"loss {final:.6f}")
print(met.metrics_report())

## Paste-back: your run

Fill this in from your own output and keep it with your notes. Nothing here should be copied from the prose above; if a field is blank, the run did not answer it.

```
chip:
torch / torch_xla versions:
PJRT_DEVICE:
step shape (batch, features):
lazy:    CompileTime / ExecuteTime / MarkStep:
eager:   EagerOpCompileTime / EagerOpExecuteTime / MarkStep:
dynamo:  CompileTime / ExecuteTime / DynamoExtractCompiledGraph:
graph hash from the IR dump:
seam log lines for two lazy steps:
one thing the report showed that the seam table did not predict:
```

In [ ]:
# Colab TPU runtime only
import json

import torch_xla.runtime as xr

chip = "unknown"
try:
    chip = xr.device_type() or chip                 # 'TPU'
except Exception:
    pass
try:
    from torch_xla._internal import tpu as _tpu
    chip = _tpu.get_tpu_env().get("ACCELERATOR_TYPE", chip)   # e.g. 'v6e-1'
except Exception:
    pass

print(json.dumps({
 "lab": "p5",
 "chip": chip,                    # cross-check against the Colab runtime badge
 "torch_xla": version("torch_xla"),
 "rows": rows,                    # the three-way diff, as measured
 "seam_log_lines": len(seam_lines),
}, indent=1))

## Mark it run

The chapter this drills is [kernels.rudrite.com/pytorch/bridges](https://kernels.rudrite.com/pytorch/bridges), whose seam lesson names the same `ComputationClient` surface you just counted calls against. Underneath it, [kernels.rudrite.com/xla/pjrt](https://kernels.rudrite.com/xla/pjrt) is `CompileAndLoad` and `ExecuteSharded` from the other side, and [kernels.rudrite.com/xla/interfaces](https://kernels.rudrite.com/xla/interfaces) walks two implementations of the interface line by line. LAB·X5 builds a third from scratch in C, which is the same boundary with your own code on the far side of it.